# Credit Risk Scorecard

## Risk Scorecard Development

The goal of this notebook is to build a transparent rule-based credit risk score using borrower and loan characteristics. Risk factors and score thresholds are based on observed default patterns identified during the exploratory risk analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/loans_clean.csv")

df.head()

,loan_id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,purpose,addr_state,dti,fico_range_low,fico_range_high,open_acc,revol_bal,revol_util,total_acc,default_flag
0,68407277,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,...,debt_consolidation,PA,5.91,675.0,679.0,7.0,2765.0,29.7,13.0,0
1,68355089,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,...,small_business,SD,16.06,715.0,719.0,22.0,21470.0,19.2,38.0,0
2,68341763,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,...,home_improvement,IL,10.78,695.0,699.0,6.0,7869.0,56.2,18.0,0
3,68476807,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,...,major_purchase,PA,25.37,695.0,699.0,12.0,21929.0,64.5,35.0,0
4,68426831,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,...,debt_consolidation,GA,10.20,690.0,694.0,5.0,8822.0,68.4,6.0,0


In [3]:
df.columns.tolist()

['loan_id',
 'loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'purpose',
 'addr_state',
 'dti',
 'fico_range_low',
 'fico_range_high',
 'open_acc',
 'revol_bal',
 'revol_util',
 'total_acc',
 'default_flag']

In [4]:
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

df[["fico_range_low", "fico_range_high", "fico_avg"]].head()

,fico_range_low,fico_range_high,fico_avg
0,675.0,679.0,677.0
1,715.0,719.0,717.0
2,695.0,699.0,697.0
3,695.0,699.0,697.0
4,690.0,694.0,692.0


In [5]:
fico_analysis = (
    df.groupby(
        pd.cut(
            df["fico_avg"],
            bins=[0, 650, 700, 750, np.inf],
            labels=["<650", "650-699", "700-749", "750+"],
            right=False
        ),
        observed=True
    )["default_flag"]
    .agg(["count", "mean"])
)

fico_analysis["default_rate"] = fico_analysis["mean"] * 100

fico_analysis[["count", "default_rate"]]

,count,default_rate
fico_avg,,
<650,2,0.000000
650-699,820825,23.592727
700-749,417917,15.665072
750+,106606,8.889744


In [6]:
dti_analysis = (
    df.groupby(
        pd.cut(
            df["dti"],
            bins=[0, 20, 35, 50, np.inf],
            labels=["<20", "20-34.99", "35-49.99", "50+"],
            right=False
        ),
        observed=True
    )["default_flag"]
    .agg(["count", "mean"])
)

dti_analysis["default_rate"] = dti_analysis["mean"] * 100

dti_analysis[["count", "default_rate"]]

,count,default_rate
dti,,
<20,807642,16.941169
20-34.99,502306,24.068795
35-49.99,32249,31.033520
50+,2777,28.700036


In [7]:
revol_analysis = (
    df.groupby(
        pd.cut(
            df["revol_util"],
            bins=[0, 30, 60, 90, np.inf],
            labels=["<30", "30-59.99", "60-89.99", "90+"],
            right=False
        ),
        observed=True
    )["default_flag"]
    .agg(["count", "mean"])
)

revol_analysis["default_rate"] = revol_analysis["mean"] * 100

revol_analysis[["count", "default_rate"]]

,count,default_rate
revol_util,,
<30,281534,15.885115
30-59.99,537540,19.942702
60-89.99,443646,21.956470
90+,81773,23.342668


## Selected Risk Factors

Based on the observed default-rate analysis, the scorecard will use three core risk factors:

- **FICO Score:** Lower FICO scores were associated with higher observed default rates.
- **Debt-to-Income Ratio (DTI):** Higher DTI levels were generally associated with higher default rates.
- **Revolving Utilization:** Default rates increased consistently as revolving utilization increased.

These factors were selected based on observed portfolio behavior rather than using arbitrary assumptions.

In [8]:
df["fico_points"] = np.select(
    [
        df["fico_avg"] < 700,
        (df["fico_avg"] >= 700) & (df["fico_avg"] < 750),
        df["fico_avg"] >= 750
    ],
    [
        30,
        15,
        0
    ],
    default=0
)

df[["fico_avg", "fico_points"]].head(10)

,fico_avg,fico_points
0,677.0,30
1,717.0,15
2,697.0,30
3,697.0,30
4,692.0,30
5,682.0,30
6,707.0,15
7,687.0,30
8,702.0,15
9,702.0,15


In [9]:
df["dti_points"] = np.select(
    [
        df["dti"] < 20,
        (df["dti"] >= 20) & (df["dti"] < 35),
        df["dti"] >= 35
    ],
    [
        0,
        15,
        30
    ],
    default=0
)

df[["dti", "dti_points"]].head(10)

,dti,dti_points
0,5.91,0
1,16.06,0
2,10.78,0
3,25.37,15
4,10.20,0
5,14.67,0
6,17.61,0
7,13.07,0
8,34.80,15
9,34.95,15


In [10]:
df["revol_points"] = np.select(
    [
        df["revol_util"] < 30,
        (df["revol_util"] >= 30) & (df["revol_util"] < 60),
        (df["revol_util"] >= 60) & (df["revol_util"] < 90),
        df["revol_util"] >= 90
    ],
    [
        0,
        10,
        20,
        30
    ],
    default=0
)

df[["revol_util", "revol_points"]].head(10)

,revol_util,revol_points
0,29.7,0
1,19.2,0
2,56.2,10
3,64.5,20
4,68.4,20
5,84.5,20
6,5.7,0
7,34.5,10
8,39.1,10
9,67.2,20


In [11]:
df["risk_score"] = (
    df["fico_points"]
    + df["dti_points"]
    + df["revol_points"]
)

df[
    [
        "loan_id",
        "fico_points",
        "dti_points",
        "revol_points",
        "risk_score"
    ]
].head(10)

,loan_id,fico_points,dti_points,revol_points,risk_score
0,68407277,30,0,0,30
1,68355089,15,0,0,15
2,68341763,30,0,10,40
3,68476807,30,15,20,65
4,68426831,30,0,20,50
5,68476668,30,0,20,50
6,67275481,15,0,0,15
7,68466926,30,0,10,40
8,68616873,15,15,10,40
9,68338832,15,15,20,50


In [12]:
df["risk_tier"] = pd.cut(
    df["risk_score"],
    bins=[-1, 20, 40, 60, 90],
    labels=["Low", "Medium", "High", "Very High"]
)

df[["loan_id", "risk_score", "risk_tier"]].head(10)

,loan_id,risk_score,risk_tier
0,68407277,30,Medium
1,68355089,15,Low
2,68341763,40,Medium
3,68476807,65,Very High
4,68426831,50,High
5,68476668,50,High
6,67275481,15,Low
7,68466926,40,Medium
8,68616873,40,Medium
9,68338832,50,High


In [13]:
tier_validation = (
    df.groupby("risk_tier", observed=True)["default_flag"]
    .agg(["count", "mean"])
)

tier_validation["default_rate"] = tier_validation["mean"] * 100

tier_validation[["count", "default_rate"]]

,count,default_rate
risk_tier,,
Low,172829,11.107511
Medium,541145,17.454102
High,436292,22.697184
Very High,195084,28.666626


## Risk Scorecard Validation

The scorecard successfully separates loans by observed default risk:

- **Low Risk:** 11.11% default rate
- **Medium Risk:** 17.45% default rate
- **High Risk:** 22.70% default rate
- **Very High Risk:** 28.67% default rate

Observed default rates increase consistently across the risk tiers, supporting the effectiveness of the rule-based scorecard for portfolio risk segmentation.

In [14]:
scorecard_df = df[
    [
        "loan_id",
        "fico_avg",
        "dti",
        "revol_util",
        "fico_points",
        "dti_points",
        "revol_points",
        "risk_score",
        "risk_tier",
        "default_flag"
    ]
]

scorecard_df.head()

,loan_id,fico_avg,dti,revol_util,fico_points,dti_points,revol_points,risk_score,risk_tier,default_flag
0,68407277,677.0,5.91,29.7,30,0,0,30,Medium,0
1,68355089,717.0,16.06,19.2,15,0,0,15,Low,0
2,68341763,697.0,10.78,56.2,30,0,10,40,Medium,0
3,68476807,697.0,25.37,64.5,30,15,20,65,Very High,0
4,68426831,692.0,10.20,68.4,30,0,20,50,High,0


In [15]:
scorecard_df.to_csv("../data/risk_scorecard.csv", index=False)

print("risk_scorecard.csv saved successfully")

risk_scorecard.csv saved successfully
